In [ ]:
import os 
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from matplotlib import pyplot as plt

# Introduction

In this notebook, we will go through:
- The basics of the self-attention mechanism
- The Multi-Head Attention mechanism
- The Transformer architecture that uses Multi-Head Attention
- Implementing a simple Transformer model in PyTorch

## Self-Attention

First, generate some random data to work with. We will generate a sequence of length $n=3$ with a key dimension $d_k=2$. 

This means that each element in the sequence is represented by a vector of length 2. 

We will generate random query, key, and value vectors for each element in the sequence.

In [ ]:
seq_len, d_k = 3, 2 # Sequence length and key dimension
# Set seed 
np.random.seed(42)
torch.manual_seed(42)

q = torch.randn(seq_len, d_k)
k = torch.randn(seq_len, d_k)
v = torch.randn(seq_len, d_k)
print("Q\n", q)
print("K\n", k)
print("V\n", v)

## 1. Compute the $\mathbf{QK}^\intercal$ product (also called the non-normalized attention logits)



In [ ]:
# Using numpy 
# [...]

# Using torch.matmul
# [...]


# Using torch.einsum
# [...]



# In the next part we will keep the calculations with torch
# [...]

## 2. Compute the attention values $\text{Attention} (\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}(\frac{\mathbf{Q}\mathbf{K}^{\intercal}}{\sqrt{d_v}})\mathbf{V}$


In [ ]:
# [...]


## 3. Create a ```scaled_dot_product``` function that computes the attention values and the attention scores given the query, key, and value tensors.

In [ ]:
def scaled_dot_product(q, k, v, mask=None):
    # [...]
    pass

# [...]

## 3. Complete the following ```MultiHeadAttention``` Class

* Initialize it properly using class inheritance
* To get the Q, K and V matrix we need to learn a projection of X. Initialize the required NN for that.
* To get the final context sensitive embedding we need to learn a projection of the output of the final MHA head. Initialize the required NN for that
* Add your ```scaled_dot_product``` as a feature of the MultiHeadAttention Class.
* Complete the ```forward()``` function.

In [ ]:
# Helper function to support different mask shapes.
# Output shape supports (batch_size, number of heads, seq length, seq length)
# If 2D: broadcasted over batch size and number of heads
# If 3D: broadcasted over number of heads
# If 4D: leave as is
def expand_mask(mask):
    assert mask.ndim >= 2, "Mask must be at least 2-dimensional with seq_length * seq_length"
    if mask.ndim == 3:
        mask = mask.unsqueeze(1)
    while mask.ndim < 4:
        mask = mask.unsqueeze(0)
    return mask

class MultiheadAttention(nn.Module):
    def __init__(self, input_dim, embedding_dim, num_heads):
        # [...]
        
        self._reset_parameters()
        
    def _reset_parameters(self):
        # Original Transformer initialization, see PyTorch documentation
        nn.init.xavier_uniform_(self.qkv_proj.weight)
        self.qkv_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.o_proj.weight)
        self.o_proj.bias.data.fill_(0)
            
    def forward(self, x, mask=None, return_attention=False):
        # batch_size, seq_len, _ = [...]

        if mask is not None:
            mask = expand_mask(mask)

        # Get query, key, value projections
        # [...]

        # Determine attention value outputs
        # [...]

        # Project output back to original dimension
        # [...]        
        
        # if return_attention:
        #     return output, attention
        # else:
        #     return output
        pass
        
# Create a MultiheadAttention layer
B, S, D = 1, 10, 768
mha = MultiheadAttention(input_dim=D, embedding_dim=D, num_heads=12)
# Test with random input
x = torch.randn(B, S, D)
output = mha(x)

## 4. Complete the following ```EncoderBlock``` class
* Initialize it correctly.
* Complete the ```forward()``` function.

In [ ]:
class EncoderBlock(nn.Module):

    def __init__(self, input_dim, num_heads, dim_feedforward, dropout=0.0):
        """
        Inputs:
            input_dim - Dimensionality of the input
            num_heads - Number of heads to use in the attention block
            dim_feedforward - Dimensionality of the hidden layer in the MLP
            dropout - Dropout probability to use in the dropout layers
        """
        super().__init__()
        
        # MHA Layer
        # [...]
        
        #Two-layer Feedforward
        self.linear = nn.Sequential(
            nn.Linear(input_dim, dim_feedforward),
            nn.Dropout(dropout),
            nn.ReLU(inplace=True),
            nn.Linear(dim_feedforward, input_dim)
        )
        
        # Normalization
        self.norm1 = nn.LayerNorm(input_dim)
        self.norm2 = nn.LayerNorm(input_dim)
        self.dropout = nn.Dropout(dropout)
        
        def forward(self, x, mask=None):
            # Attention 
            # [...]
            # Here we need to add dropout
            # [...]
            # Then we normalize
            # [...]
            
            # Then we pass it through the feedforward
            # [...]
            # Add and dropout
            # [...]
            # and again normalize
            # [...]
            
            # And we get the output of a SINGLE Encoder block
            # The transformer Encoder will have multiple of these
            return output
            

## 5. Complete this ```TransformerEncoder``` class

* Initialize it correctly.
* Complete the ```forward()``` function.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, **block_args):
        super().__init__()
        # Just stacking up the encoder blocks
        self.layers = nn.ModuleList([EncoderBlock(**block_args) for _ in range(num_layers)])
        
    def forward(self, x, mask=None):
        # [...]
        pass
    
    def get_attention_maps(self, x, mask=None):
        attention_maps = []
        for l in self.layers:
            _, attn_map = l.self_attn(x, mask=mask, return_attention=True)
            attention_maps.append(attn_map)
            x = l(x)            # Pass through the layers
            
        return attention_maps

## 6. Observe this ```PositionalEncoding``` class and try to find the links with the formulas from https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf.

In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        """
        Inputs
            d_model - Hidden dimensionality of the input.
            max_len - Maximum length of a sequence to expect.
        """
        super().__init__()

        # Create matrix of [SeqLen, HiddenDim] representing the positional encoding for max_len inputs
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        # register_buffer => Tensor which is not a parameter, but should be part of the modules state.
        # Used for tensors that need to be on the same device as the module.
        # persistent=False tells PyTorch to not add the buffer to the state dict (e.g. when we save the model)
        self.register_buffer('pe', pe, persistent=False)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

In [ ]:
encod_block = PositionalEncoding(d_model=48, max_len=96)
pe = encod_block.pe.squeeze().T.cpu().numpy()

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8,3))
pos = ax.imshow(pe, cmap="RdGy", extent=(1,pe.shape[1]+1,pe.shape[0]+1,1))
fig.colorbar(pos, ax=ax)
ax.set_xlabel("Position in sequence")
ax.set_ylabel("Hidden dimension")
ax.set_title("Positional encoding over hidden dimensions")
ax.set_xticks([1]+[i*10 for i in range(1,1+pe.shape[1]//10)])
ax.set_yticks([1]+[i*10 for i in range(1,1+pe.shape[0]//10)])
plt.show()


You can clearly see the sine and cosine waves with different wavelengths that encode the position in the hidden dimensions. Specifically, we can look at the sine/cosine wave for each hidden dimension separately, to get a better intuition of the pattern. Below we visualize the positional encoding for the hidden dimensions 1, 2, 3 and 4.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,4))
ax = [a for a_list in ax for a in a_list]
for i in range(len(ax)):
    ax[i].plot(np.arange(1,17), pe[i,:16], color=f'C{i}', marker="o", markersize=6, markeredgecolor="black")
    ax[i].set_title(f"Encoding in hidden dimension {i+1}")
    ax[i].set_xlabel("Position in sequence", fontsize=10)
    ax[i].set_ylabel("Positional encoding", fontsize=10)
    ax[i].set_xticks(np.arange(1,17))
    ax[i].tick_params(axis='both', which='major', labelsize=10)
    ax[i].tick_params(axis='both', which='minor', labelsize=8)
    ax[i].set_ylim(-1.2, 1.2)
fig.subplots_adjust(hspace=0.8)
plt.show()

## 7. Scheduler

A scheduler in machine learning is a mechanism that adjusts the learning rate during training. The learning rate is a crucial hyperparameter that controls how much to change the model in response to the estimated error each time the model weights are updated. By using a scheduler, we can start with a higher learning rate to speed up the training process and gradually decrease it to fine-tune the model. This helps in achieving better convergence and avoiding overshooting the optimal solution. Different types of schedulers, such as step decay, exponential decay, and cosine annealing, can be used depending on the specific requirements of the training process.

* Plot the scheduler's values for the given number of iterations.

In [ ]:
class CosineWarmupScheduler(torch.optim.lr_scheduler._LRScheduler):

    def __init__(self, optimizer, warmup, max_iters):
        self.warmup = warmup
        self.max_num_iters = max_iters
        super().__init__(optimizer)

    def get_lr(self):
        lr_factor = self.get_lr_factor(epoch=self.last_epoch)
        return [base_lr * lr_factor for base_lr in self.base_lrs]

    def get_lr_factor(self, epoch):
        lr_factor = 0.5 * (1 + np.cos(np.pi * epoch / self.max_num_iters))
        if epoch <= self.warmup:
            lr_factor *= epoch * 1.0 / self.warmup
        return lr_factor

In [ ]:
# Plot the learning rate schedule
# [...]